# Algoritma Baru: Sistem Absensi Cerdas

Notebook ini berisi logika utuh (jantung) dari sistem absensi baru yang kita kembangkan. Algoritma ini dirancang khusus untuk mengatasi berbagai kelemahan sistem lama, seperti pemotongan jam lintas-hari (shift malam) dan masalah double-tap.

In [ ]:
import pandas as pd
import re


## 1. Fungsi Parser Data Mentah

Fungsi ini bertugas mengambil data murni dari mesin (log jari menempel). Data ini benar-benar mentah tanpa ada rekayasa atau asumsi apa pun.

In [ ]:
def parse_sql_dump(filepath):
    print(f"Membaca file {filepath}...")
    data = []
    
    # Menangkap Kode Area (kolom 1), PIN (kolom 2), Waktu_Scan (kolom 3)
    pattern = re.compile(r"\('([^']*)',\s*'([^']*)',\s*'([^']*)',\s*(\d+),\s*'([^']*)'\)")
    
    try:
        with open(filepath, 'r', encoding='utf-8') as file:
            for line in file:
                if line.strip().startswith("('"):
                    matches = pattern.findall(line)
                    for match in matches:
                        kode_area = match[0]
                        pin = match[1]
                        waktu_scan = match[2]
                        if pin and waktu_scan:
                            data.append({
                                'Kode_Area': kode_area,
                                'PIN': pin, 
                                'Waktu_Scan': waktu_scan
                            })
    except Exception as e:
        print(f"Gagal membaca {filepath}: {e}")
    return pd.DataFrame(data)


## 2. Fungsi Deteksi Shift Otomatis

Algoritma ini jauh lebih fleksibel dari sistem lama. Sistem mendeteksi shift hanya dari **Jam Kedatangan (IN)** dengan rentang toleransi yang sangat lebar untuk mengakomodir karyawan yang datang lebih awal (Lembur Maju).

In [ ]:
def tentukan_shift(jam_masuk):
    if pd.isna(jam_masuk) or jam_masuk == "-":
        return "Tidak Diketahui"
        
    jam = int(str(jam_masuk)[:2])
    
    # Karyawan tidak mungkin telat berjam-jam, tapi bisa absen lebih awal
    # Shift 1 (08:00-16:00): Masuk jam 05:00 s/d 10:59
    if 5 <= jam < 11:
        return "Shift 1 (08:00-16:00)"
    # Shift 2 (16:00-00:00): Masuk jam 11:00 s/d 18:59
    elif 11 <= jam < 19:
        return "Shift 2 (16:00-00:00)"
    # Shift 3 (00:00-08:00): Masuk jam 19:00 s/d 02:59
    elif jam >= 19 or jam < 3:
        return "Shift 3 (00:00-08:00)"
    else:
        return "Shift Tidak Sesuai Jadwal"


## 3. Logika Utama: Sequential Pairing & Multi-Factory Clustering

Di sinilah keajaibannya terjadi. Terdapat dua pelindung utama:
1. **Double-Tap Protection**: Karyawan yang absen 2 kali di jam yang sama otomatis dibersihkan (jarak antar absen minimal 1 jam).
2. **Multi-Factory Grouping**: Pasangan masuk-keluar dihitung per Lokasi (Kode Area). Menjamin orang yang berpindah pabrik datanya tidak tertukar.
3. **Anti-Cutoff (Lintas Hari)**: Shift malam dicari pasangannya keesokan harinya selama durasinya masih logis (Maksimal 17 Jam).

In [ ]:
def proses_pairing():
    df = parse_sql_dump('t_absensi_solutions_fp.sql')
    if df.empty:
        return None
        
    df['Waktu_Scan'] = pd.to_datetime(df['Waktu_Scan'])
    df = df.sort_values(['PIN', 'Waktu_Scan'])
    
    # -------------------------------------------------------------
    # PEMBERSIHAN DATA: Hapus Double Scan (Selisih < 1 jam)
    # -------------------------------------------------------------
    df['Prev_Scan'] = df.groupby(['PIN', 'Kode_Area'])['Waktu_Scan'].shift(1)
    df['Selisih_Jam'] = (df['Waktu_Scan'] - df['Prev_Scan']).dt.total_seconds() / 3600
    df = df[df['Selisih_Jam'].isna() | (df['Selisih_Jam'] >= 1.0)].copy()
    
    records = []
    # KELOMPOKKAN BERDASARKAN PIN DAN KODE AREA (Solusi lintas-pabrik)
    for (pin, kode_area), group in df.groupby(['PIN', 'Kode_Area']):
        scans = group['Waktu_Scan'].tolist()
        areas = group['Kode_Area'].tolist()
        
        i = 0
        while i < len(scans):
            in_time = scans[i]
            in_area = areas[i]
            out_time = pd.NaT
            out_area = "-"
            
            # Sequential Pairing (Mencari scan pulang)
            if i + 1 < len(scans):
                duration = (scans[i+1] - in_time).total_seconds() / 3600
                if duration <= 17:
                    out_time = scans[i+1]
                    out_area = areas[i+1]
                    i += 2
                else:
                    # Karyawan lupa absen pulang, sisa jam dianggap gantung
                    i += 1
            else:
                i += 1
                
            # Pemberian Label Keterangan Otomatis
            keterangan = "Normal"
            if pd.isna(out_time):
                keterangan = "Lupa Absen Keluar (Input Manual)"
            else:
                dur = (out_time - in_time).total_seconds() / 3600
                if dur > 9:
                    jam_l = int(dur - 8)
                    keterangan = f"Ada Lembur (+/- {jam_l} Jam)"
                
            records.append({
                'PIN': pin,
                'Tanggal': in_time.date(),
                'Kode_Area': str(in_area),
                'Jam_Masuk': in_time,
                'Jam_Keluar': out_time,
                'Keterangan': keterangan
            })
            
    df_rekap = pd.DataFrame(records)
    return df_rekap


Jalankan perintah di bawah ini untuk melihat hasil akhir tabel (*Output* sudah siap langsung diolah HRD).

In [ ]:
# df_final = proses_pairing()
# print(df_final.head())